In [1]:
import cv2
import torch
import torchvision.transforms as transforms
import numpy as np
import math
from torchvision.models.detection import ssdlite320_mobilenet_v3_large

In [2]:
# Load the pre-trained model
model = ssdlite320_mobilenet_v3_large(pretrained=True)
model.eval()

C:\Users\HP\anaconda3\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\HP\anaconda3\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1`. You can also use `weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


SSD(
  (backbone): SSDLiteFeatureExtractorMobileNet(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (1): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            )
          )
        )
        (2): Invert

In [3]:
# Initialize video capture
cap = cv2.VideoCapture('C:/Users/HP/Py Code/Neural Network/Pytorch/Object detection/v1.mp4')

In [4]:
# Tracker class to track the detected person objects based on the proximity between center points of bounding boxes
class Tracker:
    def __init__(self):
        # Store the center positions of the objects
        self.center_points = {}
        # Keep the count of the IDs
        # each time a new object is detected, the count will increase by one
        self.id_count = 0

    def update(self, objects_rect):
        # Objects' bounding boxes and IDs
        objects_bbs_ids = []

        # Get the center point of each new object
        for rect in objects_rect:
            x, y, w, h = rect
            cx = (x + w) // 2
            cy = (y + h) // 2

            # Check if the object was detected already
            same_object_detected = False
            for id, pt in self.center_points.items():
                dist = math.hypot(cx - pt[0], cy - pt[1])

                if dist < 35:
                    self.center_points[id] = (cx, cy)
                    objects_bbs_ids.append([x, y, w, h, id])
                    same_object_detected = True
                    break

            # If it's a new object, assign a new ID
            if not same_object_detected:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1

        # Clean the dictionary to remove IDs not used anymore
        new_center_points = {}
        for obj_bb_id in objects_bbs_ids:
            _, _, _, _, object_id = obj_bb_id
            center = self.center_points[object_id]
            new_center_points[object_id] = center

        # Update the dictionary with IDs not used removed
        self.center_points = new_center_points.copy()
        return objects_bbs_ids


# Define a mouse event callback function
def POINTS(event, x, y, flags, param):
    if event == cv2.EVENT_MOUSEMOVE:
        colorsBGR = [x, y]
        print(colorsBGR)


cv2.namedWindow('FRAME')
cv2.setMouseCallback('FRAME', POINTS)

# Define the Tracker
tracker = Tracker()

# Define the region of interest polygon
area_1 = [(3, 2), (1000, 2), (1000, 500), (3, 500)]

area1 = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (1020, 500))

    # Draw the region of interest polygon
    cv2.polylines(frame, [np.array(area_1, np.int32)], True, (0, 255, 0), 3)

    # Convert the frame to a tensor
    img = transforms.ToTensor()(frame).unsqueeze(0)

    # Perform object detection
    with torch.no_grad():
        outputs = model(img)

    # Extract bounding boxes, labels, and scores for each detection
    detections = outputs[0]['boxes']
    labels = outputs[0]['labels']
    scores = outputs[0]['scores']
    
    detected_objects = []
    for idx, label in enumerate(labels):
        if label == 1 and scores[idx] > 0.5:  # Considering 'person' label with a score above 0.5
            box = detections[idx]
            x1, y1, x2, y2 = [int(coord.item()) for coord in box]
            detected_objects.append([x1, y1, x2 - x1, y2 - y1])

    # Update the tracker with the detected bounding boxes of 'person' objects
    boxes_ids = tracker.update(detected_objects)

    area1.clear()
    for box_id in boxes_ids:
        x, y, w, h, id = box_id
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 255), 2)
        cv2.putText(frame, str(id), (x, y), cv2.FONT_HERSHEY_PLAIN, 1, (255, 0, 0), 2)

        # Check if the 'person' object is within the defined region of interest
        result = cv2.pointPolygonTest(np.array(area_1, np.int32), (x, y), False)
        if result > 0:
            area1.add(id)

    p = len(set(id for _, _, _, _, id in boxes_ids))
    print(p)
    cv2.putText(frame, 'count:' + str(p), (20, 30), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)
    if p > 5:
        cv2.putText(frame, 'Overloaded', (20, 60), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)

    cv2.imshow('FRAME', frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


3
3
3
3
3
2
2
1
1
1
1
1
1
1
1
[677, 255]
1
1
1
1
1
1
1
2
2
2
2
2
2
2
2
1
3
3
2
2
2
3
3
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
